# Breast Cancer Classification Project
**Course:** Applied Machine Learning — Basic Track  
**Dataset:** Breast Cancer Wisconsin (Diagnostic)  
**Task:** Classify tumors as malignant or benign using machine learning.

## Goal
The goal of this project is to build machine learning models that can predict whether a breast tumor is **malignant** (cancerous) or **benign** (not cancerous), based on measurements taken from cell samples.

## Dataset
The dataset contains **569 patient samples** and **30 features** describing cell nucleus properties such as radius, texture, and smoothness. It is available directly from the scikit-learn library.

## Approach
We follow the standard machine learning workflow:
1. Load and explore the data
2. Preprocess the data
3. Train and compare models
4. Evaluate results
5. Interpret findings

---
**LLM Usage Declaration:** LLMs were used for code debugging and language polishing. All methodological choices, results, and interpretations were reviewed and validated by the author.

## Section 1 — Import Libraries

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset
from sklearn.datasets import load_breast_cancer

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('Libraries loaded successfully.')

## Section 2 — Load the Dataset

In [ ]:
# Load the dataset from sklearn (no download needed)
data = load_breast_cancer()

# Create a DataFrame (table) for easier exploration
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # 0 = malignant, 1 = benign

# Basic information
print('Number of samples :', df.shape[0])
print('Number of features:', df.shape[1] - 1)
print('Classes           :', list(data.target_names))
print('Missing values    :', df.isnull().sum().sum())
print()
print('First 5 rows:')
df.head()

In [ ]:
# Check class distribution
benign_count    = (df['target'] == 1).sum()
malignant_count = (df['target'] == 0).sum()

print('Class distribution:')
print(f'  Benign    : {benign_count} samples ({benign_count/len(df)*100:.1f}%)')
print(f'  Malignant : {malignant_count} samples ({malignant_count/len(df)*100:.1f}%)')
print()
print('The dataset is slightly imbalanced — more benign than malignant samples.')
print('This means we should not rely on accuracy alone; we will also check Recall and F1-score.')

## Section 3 — Exploratory Data Analysis (EDA)

Before building any model, we explore the data to understand its structure, spot patterns, and identify challenges.

In [ ]:
# Plot 1: Class Distribution
labels = ['Benign', 'Malignant']
counts = [benign_count, malignant_count]

plt.figure(figsize=(6, 4))
plt.bar(labels, counts, color=['steelblue', 'tomato'], edgecolor='black')
plt.title('Class Distribution')
plt.xlabel('Diagnosis')
plt.ylabel('Number of Samples')
for i, v in enumerate(counts):
    plt.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print('The dataset has more benign samples (357) than malignant (212).')
print('This mild imbalance means we need to check Recall, not just accuracy.')

In [ ]:
# Plot 2: Feature distributions for the 3 most informative features
# We pick radius mean, concave points mean, and area mean
# because they are known to differ most between benign and malignant

features_to_plot = ['mean radius', 'mean concave points', 'mean area']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, feature in enumerate(features_to_plot):
    benign_vals    = df[df['target'] == 1][feature]
    malignant_vals = df[df['target'] == 0][feature]
    axes[i].hist(benign_vals,    bins=20, alpha=0.6, color='steelblue', label='Benign')
    axes[i].hist(malignant_vals, bins=20, alpha=0.6, color='tomato',    label='Malignant')
    axes[i].set_title(feature)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')
    axes[i].legend()

plt.suptitle('Feature Distributions: Benign vs Malignant', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Observation: Malignant tumors have higher values for radius, area, and concave points.')
print('These features clearly separate the two classes, making them useful for classification.')

In [ ]:
# Plot 3: Correlation heatmap (only mean features for readability)
mean_features = [col for col in df.columns if 'mean' in col]

plt.figure(figsize=(10, 8))
sns.heatmap(
    df[mean_features].corr(),
    annot=True, fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    annot_kws={'size': 8}
)
plt.title('Correlation Between Mean Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Observation: Some features are highly correlated (e.g. radius, perimeter, and area).')
print('This is expected — they all measure tumor size but in different ways.')
print('Tree-based models handle correlated features well without extra treatment.')

## Section 4 — Preprocessing

We prepare the data for training:
- **Split** into training and test sets so we can evaluate on unseen data
- **Scale** features so all values are on the same range (important for KNN and Logistic Regression)

In [ ]:
# Separate features (X) from the target label (y)
X = df.drop(columns=['target'])
y = df['target']

# Split: 80% for training, 20% for testing
# stratify=y keeps the same class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')

In [ ]:
# Scale features using StandardScaler
# This makes each feature have mean=0 and standard deviation=1
# We fit ONLY on training data to avoid data leakage

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform on train
X_test_scaled  = scaler.transform(X_test)        # only transform on test

print('Scaling done.')
print('Important: the scaler was fitted only on training data, not on test data.')

## Section 5 — Model Training & Comparison

We train four models and compare their performance. We start with a simple model (Logistic Regression) as a baseline and move to more complex ones.

In [ ]:
# Define the four models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42)
}

# Train each model and record its accuracy
results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)          # train
    y_pred = model.predict(X_test_scaled)        # predict
    acc = accuracy_score(y_test, y_pred)         # evaluate
    results.append({'Model': name, 'Accuracy': round(acc, 4)})
    print(f'{name:<25} Accuracy: {acc:.4f}')

results_df = pd.DataFrame(results).set_index('Model')

In [ ]:
# Plot: Model Accuracy Comparison
plt.figure(figsize=(8, 5))
plt.bar(
    results_df.index,
    results_df['Accuracy'],
    color=['steelblue', 'seagreen', 'orange', 'mediumpurple'],
    edgecolor='black'
)
plt.title('Model Accuracy Comparison', fontsize=13, fontweight='bold')
plt.xlabel('Model')
plt.ylabel('Accuracy')
plt.ylim(0.85, 1.0)
plt.xticks(rotation=15)
for i, v in enumerate(results_df['Accuracy']):
    plt.text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## Section 6 — Evaluation

We evaluate the best model in detail using:
- **Confusion Matrix** — shows correct and incorrect predictions per class
- **Classification Report** — shows Precision, Recall, and F1-score per class

In [ ]:
# Find the best model by accuracy
best_name  = results_df['Accuracy'].idxmax()
best_model = models[best_name]
y_pred_best = best_model.predict(X_test_scaled)

print(f'Best model: {best_name}')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_best):.4f}')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True, fmt='d',
    cmap='Blues',
    xticklabels=['Malignant', 'Benign'],
    yticklabels=['Malignant', 'Benign'],
    linewidths=0.5
)
plt.title(f'Confusion Matrix — {best_name}', fontsize=12, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

print('Rows = Actual class | Columns = Predicted class')
print('The diagonal cells show correct predictions.')
print('Off-diagonal cells show errors.')

In [ ]:
# Classification Report
print(f'Classification Report — {best_name}')
print()
print(classification_report(
    y_test, y_pred_best,
    target_names=['Malignant', 'Benign']
))

print('Recall for Malignant is the most important metric here.')
print('A low recall means the model is missing actual cancer cases — the most dangerous error.')

## Section 7 — Conclusion

### Summary of Results

In [ ]:
# Print final comparison table
print('Model Accuracy Summary:')
print(results_df.to_string())
print()
print(f'Best performing model: {best_name}')

### Interpretation

In this project, we applied a complete machine learning workflow to classify breast tumors as malignant or benign.

**Data exploration** revealed that the dataset has a mild class imbalance (63% benign, 37% malignant). Features related to tumor size (radius, area) and shape irregularity (concave points) show the clearest separation between classes. Some features are highly correlated, which is expected since they measure related properties of the same tumor.

**Preprocessing** included an 80/20 train-test split with stratification, and feature scaling using StandardScaler. Scaling was applied only after splitting to avoid data leakage.

**Model comparison** showed that all four models achieved high accuracy on this dataset, which confirms that the features are strongly informative. Random Forest performed best overall due to its ability to handle correlated features and capture nonlinear patterns.

**Evaluation** using Precision, Recall, and F1-score showed that the best model correctly identifies most malignant cases. In a medical setting, **Recall for malignant tumors** is the most important metric — missing a cancer case (False Negative) is far more dangerous than a false alarm.

**Recommendation:** Random Forest is the most suitable model for this task. Future work could include hyperparameter tuning and cross-validation for more reliable performance estimates.